## Writing `docx` Documents

Below is an example on how to use pydocmaker to write word docx documents from format templates
and also automatically "replace" fields (MergeFields in Word or plain text) to be filled out in the docx document with text from python.

**NOTE**: Updating word documents and exporting them to PDF requires the `win32com` api, Microsoft Word installed and only works on Windows.

**NOTE**: Exporting word documents is unfortunately very slow, but hey... it works :-)




In [1]:

import pydocmaker as pyd
import os
print(pyd.__version__)

2.6.9


In [2]:
# get a pyd example document to show the concept
doc = pyd.get_example()

In [3]:
# some dir to write my example files to
os.makedirs('../exported_docs_examples', exist_ok=True)

### To docx (no template, no dependencies)

In [4]:
# as bytes
bts = doc.to_docx()
type(bts), len(bts)

(bytes, 14144)

In [5]:
bts

b'PK\x03\x04\x14\x00\x02\x08\x08\x00\xb1p\xca\\\xaf9I\x01\x99\x01\x00\x00\xd2\x07\x00\x00\x13\x00\x00\x00[Content_Types].xml\xb5\x95\xd1N\xc20\x14\x86_e\xd9\xada\x05\x12\x8d1\x80\x17*\x897j">@i\xcf\xa0q\xedi\xda3\xd4\xb7\xf7l\x93\xc5\x10\xddP\xe0f\xc9z\xce\xf9\xfe\xbf\xff\xdalr\xfdn\x8bd\x03!\x1at\xd3t\x94\r\xd3\x04\x9cBm\xdcj\x9a\xbe,\xe6\x83\xcb\xf4z6Y|x\x88\t\xb7\xba8M\xd7D\xfeJ\x88\xa8\xd6`e\xcc\xd0\x83\xe3J\x8e\xc1J\xe2\xd7\xb0\x12^\xaaW\xb9\x021\x1e\x0e/\x84BG\xe0h@\x15#\x9dMn!\x97eA\xc9\xdd;/7\xb2<\x9e&7M_%5M\xa5\xf7\x85Q\x92\xb8,\xea\xaa\xf8q0@\x11;&7N\xef\xd8\x1b|Y\xcbx\xb2\xee\x89k\xe3\xe3Y\x87\x04j\xa2\xfco\x1a\x98\xe7F\x81FUZ\x1e\xc9p\x99\x97\x91\xbbA\xcf\x19R\xeb<r\xe2\xc1hH\x9ed\xa0\x07i\x99)\xde0h\xf1\x06\xcbg \xe2\xf4c\xd6\x9dJ\xbfn\x05\xf4\x01\x15\xc4\xc8<[d\xdf\xe0\xed\x8e\x7fu\xe2J\xbb\x84\xc0\xbd\xc7\xf7\xd1\xa2\xfb]\xc4\x93\x85\x11\xf7N\x82\xf8\x9cC\xf3\x1c\x1d\xec\xa3\xc6\xf4k\xe6\xac\xb0\x90\xcb\x02\x8e\xbf\xf1\x16\xdd\xe9\x82\xe7\x9f\x02\xfa(X\xed`\x0fP]\'\rz\xc

In [6]:
# write to file
file_exists = doc.to_docx("../exported_docs_examples/outfile_raw.docx")
file_exists

True

### Prepare a template and fields (optional)

prepare, a template and some fields in the template. 

In [7]:
templatepath = os.path.join(pyd.get_registered_template_dirs()[0], 'word_template_with_mergefields.docx')
templatepath, os.path.exists(templatepath)

('C:\\Users\\tglaubach\\repos\\pydocmaker\\src\\pydocmaker\\templates\\word_template_with_mergefields.docx',
 True)

In [8]:
# HOWTO: 
#  Adding MergeFields In Word to replace them later: 
#    Go to Insert -> Quick Parts -> Field -> MergeField.

metadata = {
    'repno': "1234",
    "summary": "This is a nice workflow for automatically creating docx documents",
    "date": "2025-12-13",
    "comment": f"this is my comment!",
    "author": "Me"
}


this is the quick and easy way using the common pydocmaker api:

### To docx (no additional dependencies)

In [9]:
# three different examples below
file_exists = doc.to_docx("../exported_docs_examples/outfile.docx", template=templatepath, template_params=metadata)
file_exists


True

### To docx via Word (with additional dependencies)

This will also update all fields and the table of contents, but will need win32com for it.

In [10]:
file_exists = doc.to_docx("../exported_docs_examples/outfile_w32.docx", template=templatepath, template_params=metadata, use_w32=True)
file_exists

NotADirectoryError: [WinError 267] The directory name is invalid: 'C:\\Users\\TGLAUB~1\\AppData\\Local\\Temp\\tmp1dux72d3\\outfile_w32.docx'

### To PDF via docx with Word (with additional dependencies)

This will do the same as the last step, but also export the file to PDF after it has created the `.docx` file. 

In [ ]:
file_exists = doc.to_docx("../exported_docs_examples/outfile_w32.pdf", template=templatepath, template_params=metadata, use_w32=True, as_pdf=True, compress_images=True)
file_exists

### To PDF via docx with Libreoffice (with additional dependencies)

This will instead use the Libreoffice software to convert a docx file to PDF.

In [ ]:
file_exists = doc.to_docx("../exported_docs_examples/outfile_libreoffice.pdf", template=templatepath, template_params=metadata, as_pdf=True)
file_exists

### The hard way: using the low level classes and functions

you can also work with the exporting classes directly to get more control:

In [ ]:
outpath = '../exported_docs_examples/outfile_lowlevel.docx'
outpath_pdf = outpath.replace(".docx", ".pdf")

docxf = pyd.DocxFile(templatepath).replace_fields(metadata).append(doc.to_docx())
docxf.save(outpath)

if pyd.DocxFileW32.is_installed():
    with pyd.DocxFileW32(outpath) as docxw32f:
        docxw32f.update_fields()
        docxw32f.compress_images()
        docxw32f.export( outpath_pdf )

os.path.exists(outpath), os.path.exists(outpath_pdf)